# 🏆 IFest DAC 2026 — Data Analysis Competition (Penyisihan)
## Deteksi Inkongruensi Berita Bahasa Indonesia (Macro F1-Score)

**Pipeline Opsi A: Murni Algoritmik & Patuh Regulasi (Discourse-Aware NLP, Adaptive Inversion Scopes, & Multi-Model Stacking Ensemble)**

Notebook ini mengimplementasikan solusi klasifikasi biner untuk mendeteksi ketidaksesuaian (*incongruence / clickbait*) antara judul dan isi berita berbahasa Indonesia. Solusi ini dirancang secara murni berbasis rekayasa 45 fitur diskursus-leksikal adaptif dan pembelajaran mesin ensemble klasik yang mematuhi secara ketat seluruh regulasi resmi kompetisi:
1. **Rule #4**: *No Pretrained Models* (bebas IndoBERT, Word2Vec/FastText pretrained, dll.).
2. **Rule #5**: *No Generative AI / LLM APIs* (bebas OpenAI, Gemini, Claude, dll.).
3. **Rule #8**: *No Data Leakage* (fitur statistik & LSA di-fit secara ketat HANYA pada data latih; evaluasi per-sampel murni adaptif terhadap data baru).
4. **Rule #11**: *Deterministic & Reproducible* (`SEED = 42`).

In [1]:
import os
import random
import re
import math
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

# Menetapkan Random Seed untuk Reproducibility (Kewajiban Guidebook Hal. 5 Poin 6 & Rule #11)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("All libraries imported successfully! Deterministic Random Seed set to 42.")


All libraries imported successfully! Deterministic Random Seed set to 42.


## 1. Load Data & Eksplorasi Distribusi
Dataset terdiri dari 14.397 data latih (`train.csv`) dan 3.603 data uji (`test.csv`).

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

print("\n--- Distribusi Label Train ---")
print(train['label'].value_counts())
print(f"Rasio Class 1: {(train['label']==1).mean()*100:.2f}%")
print(f"Rasio Class 0: {(train['label']==0).mean()*100:.2f}%")

Train shape: (14397, 4)
Test shape:  (3603, 3)

--- Distribusi Label Train ---
label
1    12960
0     1437
Name: count, dtype: int64
Rasio Class 1: 90.02%
Rasio Class 0: 9.98%


## 2. Kamus Leksikal, Entitas Kanonikal, & Skop Negasi Adaptif
Modul ini mendefinisikan kamus leksikal terkurasi untuk menangani variasi penulisan bahasa Indonesia:
- **Kamus Lokasi & Normalisasi**: Menangani singkatan wilayah (*jabar*, *jakpus*, *diy*, dll.) ke bentuk kanonikal.
- **Kamus Tokoh Publik & Kanonikalisasi Alias**: Menghubungkan alias (*jokowi*, *prabowo*, *terawan*, dll.) ke nama standar.
- **Daftar Lembaga & Penanda Entitas Komposit**: Mengenali kementerian, institusi riset/kampus (*ugm*, *ui*, *kemenkes*, dll.) dan penanda institusi wilayah (*bea cukai*, *rsud*, *polres*, dll.).
- **Indikator Negasi, Verba Inheren, & Antonim**: Menangkap pembalikan makna faktual (*tak*, *belum*, *tolak*, dll.), verba negatif (*melarang*, *membatalkan*, *menggagalkan*, dll.), dan pasangan antonim polaritas.
- **Detektor Sanggahan & Unit Angka**: Mengidentifikasi leksikon sanggahan berita (*Riedel et al.*) dan pasangan angka beserta satuannya.

In [3]:
import re

# 1. Indonesian Stopwords (Kompak & Informatif)
STOP_WORDS = {
    'yang', 'di', 'dan', 'ini', 'dari', 'dalam', 'untuk', 'pada', 'ke', 'karena',
    'oleh', 'saat', 'ada', 'akan', 'juga', 'dengan', 'adalah', 'ia', 'itu', 'atau',
    'bisa', 'tersebut', 'bukan', 'tidak', 'tak', 'namun', 'mereka', 'kita', 'para',
    'lagi', 'saja', 'sudah', 'secara', 'harus', 'hanya', 'agar', 'bagi', 'bahwa',
    'terus', 'setelah', 'terkait', 'tentang', 'atas', 'olehnya', 'menjadi', 'sebagai',
    'masih', 'bila', 'belum', 'banyak', 'sangat', 'tahun', 'hari', 'bulan', 'lalu',
    'seperti', 'antara', 'dapat', 'mengatakan', 'ujar', 'kata', 'ungkap', 'tambah',
    'jelas', 'pungkas', 'tutur', 'sebut', 'beber', 'imbuh'
}

# 2. Pasangan Antonim & Konflik Semantik
EXPANDED_ANTONYM_PAIRS = [
    ('positif', 'negatif'), ('naik', 'turun'), ('lonjakan', 'penurunan'),
    ('buka', 'tutup'), ('dibuka', 'ditutup'), ('tambah', 'kurang'),
    ('tinggi', 'rendah'), ('meningkat', 'menurun'), ('sembuh', 'meninggal'),
    ('hidup', 'mati'), ('masuk', 'keluar'), ('zona merah', 'zona hijau'),
    ('sukses', 'gagal'), ('untung', 'rugi'), ('kalah', 'menang'),
    ('benar', 'salah'), ('palsu', 'asli'), ('hoaks', 'fakta')
]

# 3. Leksikon Sanggahan (Refuting Words - Riedel et al. FNC-1)
REFUTING_PHRASES = [
    'tidak benar', 'tak benar', 'bukan', 'keliru', 'hoaks', 'hoax',
    'bohong', 'bantah', 'membantah', 'dibantah', 'fiktif', 'palsu',
    'disinformasi', 'misinformasi', 'manipulasi', 'klaim sepihak'
]
re_refuting = re.compile(r'\b(?:' + '|'.join(re.escape(w) for w in REFUTING_PHRASES) + r')\b', re.I)
re_num_unit = re.compile(r'(\b\d+[\d\.,]*)\s+([a-zA-Z]+)')

# 4. Kamus Wilayah / Lokasi & Kanonikalisasi Alias
CANONICAL_LOC = {
    'jabar': 'jawa barat', 'jateng': 'jawa tengah', 'jatim': 'jawa timur',
    'diy': 'yogyakarta', 'jogja': 'yogyakarta', 'jogjakarta': 'yogyakarta',
    'dki': 'dki jakarta', 'jakpus': 'jakarta pusat', 'jakbar': 'jakarta barat',
    'jaksel': 'jakarta selatan', 'jaktim': 'jakarta timur', 'jakut': 'jakarta utara',
    'surakarta': 'solo', 'ina': 'indonesia', 'ri': 'indonesia'
}

ALL_LOCATIONS = [
    'deli serdang', 'tebing tinggi', 'pematang siantar', 'pematangsiantar', 'tanjung balai', 'tanjungbalai',
    'bandar lampung', 'padang panjang', 'pesisir selatan', 'tanah datar', 'pagar alam', 'ogan ilir',
    'muara enim', 'banyuasin', 'kutai kartanegara', 'sulawesi tenggara', 'sulawesi selatan', 'sulawesi utara',
    'sulawesi tengah', 'sulawesi barat', 'sumatera utara', 'sumatera barat', 'sumatera selatan',
    'jawa timur', 'jawa barat', 'jawa tengah', 'nusa tenggara barat', 'nusa tenggara timur',
    'kalimantan timur', 'kalimantan barat', 'kalimantan selatan', 'kalimantan tengah', 'kalimantan utara',
    'dki jakarta', 'luwu utara', 'luwu timur', 'konawe utara', 'konawe selatan', 'musi banyuasin',
    'toba samosir', 'gunung kidul', 'gunungkidul', 'kulon progo', 'kulonprogo',
    'jakarta pusat', 'jakarta barat', 'jakarta selatan', 'jakarta timur', 'jakarta utara',
    'jakpus', 'jakbar', 'jaksel', 'jaktim', 'jakut',
    'jakarta', 'jabar', 'jateng', 'jatim', 'banten', 'jogja', 'yogyakarta', 'diy',
    'bandung', 'surabaya', 'semarang', 'bogor', 'depok', 'tangerang', 'bekasi',
    'solo', 'surakarta', 'malang', 'serang', 'cilegon', 'pandeglang', 'lebak',
    'sukabumi', 'cianjur', 'purwakarta', 'karawang', 'subang', 'indramayu',
    'cirebon', 'majalengka', 'sumedang', 'kuningan', 'ciamis', 'tasikmalaya',
    'garut', 'banjar', 'pangandaran', 'cimahi', 'brebes', 'tegal', 'slawi',
    'pemalang', 'pekalongan', 'batang', 'kendal', 'demak', 'kudus', 'jepara',
    'pati', 'rembang', 'blora', 'grobogan', 'purwodadi', 'sragen', 'karanganyar',
    'wonogiri', 'sukoharjo', 'klaten', 'boyolali', 'magelang', 'muntilan',
    'temanggung', 'wonosobo', 'purworejo', 'kebumen', 'banyumas', 'purwokerto',
    'purbalingga', 'banjarnegara', 'cilacap', 'salatiga', 'sleman', 'bantul',
    'wates', 'wonosari', 'pacitan', 'ponorogo',
    'trenggalek', 'tulungagung', 'blitar', 'kediri', 'nganjuk', 'madiun',
    'magetan', 'ngawi', 'bojonegoro', 'tuban', 'lamongan', 'gresik', 'bangkalan',
    'sampang', 'pamekasan', 'sumenep', 'sidoarjo', 'mojokerto', 'jombang',
    'pasuruan', 'probolinggo', 'lumajang', 'bondowoso', 'situbondo', 'jember',
    'banyuwangi', 'batu',
    'aceh', 'sumut', 'sumbar', 'riau', 'kepri', 'jambi', 'bengkulu', 'sumsel',
    'babel', 'lampung', 'medan', 'padang', 'palembang', 'pekanbaru', 'batam',
    'tanjungpinang', 'metro', 'pangkalpinang', 'binjai', 'asahan',
    'karo', 'simalungun', 'tapanuli', 'toba', 'mandailing', 'langkat', 'nias',
    'labuhanbatu', 'lhokseumawe', 'langsa', 'sabang', 'pidie', 'meulaboh',
    'bukittinggi', 'payakumbuh', 'pariaman', 'solok', 'sawahlunto', 'agam',
    'pasaman', 'dumai', 'bengkalis', 'siak', 'kampar', 'rokan', 'indragiri',
    'pelalawan', 'bintan', 'karimun', 'natuna', 'anambas', 'lingga', 'prabumulih',
    'lubuklinggau', 'lahat', 'musi', 'bangka', 'belitung',
    'kalbar', 'kalteng', 'kalsel', 'kaltim', 'kaltara', 'pontianak', 'singkawang',
    'palangkaraya', 'banjarmasin', 'banjarbaru', 'samarinda', 'balikpapan',
    'bontang', 'tarakan', 'sambas', 'sintang', 'ketapang', 'kapuas', 'kotawaringin',
    'barito', 'kutai', 'berau', 'pasir', 'penajam', 'nunukan', 'bulungan', 'malinau',
    'sulsel', 'sulut', 'sulteng', 'sultra', 'sulbar', 'gorontalo', 'makassar',
    'manado', 'palu', 'kendari', 'mamuju', 'parepare', 'palopo',
    'bitung', 'tomohon', 'kotamobagu', 'bau-bau', 'gowa', 'maros', 'bone',
    'luwu', 'wajo', 'soppeng', 'bulukumba', 'bantaeng', 'selayar', 'toraja',
    'enrekang', 'sidrap', 'pinrang', 'minahasa', 'sangihe', 'talaud', 'poso',
    'donggala', 'banggai', 'morowali', 'tolitoli', 'buol', 'buton', 'muna',
    'kolaka', 'konawe', 'wakatobi',
    'bali', 'denpasar', 'badung', 'gianyar', 'tabanan', 'buleleng', 'jembrana',
    'karangasem', 'klungkung', 'bangli', 'mataram', 'bima', 'lombok', 'sumbawa',
    'dompu', 'kupang', 'timor', 'belu', 'flores', 'ende', 'manggarai', 'sumba',
    'alor', 'rote', 'ambon', 'tual', 'ternate', 'tidore', 'seram', 'buru', 'halmahera',
    'jayapura', 'sorong', 'manokwari', 'merauke', 'mimika', 'timika', 'biak',
    'nabire', 'fakfak', 'wamena', 'papua',
    'amerika', 'inggris', 'jerman', 'prancis', 'china', 'cina', 'jepang', 'korsel',
    'brasil', 'brazil', 'meksiko', 'italia', 'spanyol', 'rusia', 'afsel', 'afrika',
    'australia', 'singapura', 'malaysia', 'filipina', 'thailand', 'vietnam', 'india',
    'arab', 'saudi', 'turki', 'iran', 'irak', 'israel', 'palestina', 'mesir',
    'belanda', 'portugal', 'kanada', 'selandia', 'ukraina', 'lebanon', 'indonesia', 'ri'
]
LOC_REGEX = re.compile(r'\b(' + '|'.join(re.escape(loc) for loc in sorted(ALL_LOCATIONS, key=len, reverse=True)) + r')\b', re.I)

# 5. Kamus Tokoh Publik & Kanonikalisasi Alias
CANONICAL_FIGURES = {
    'joko widodo': 'jokowi', 'maruf amin': 'ma ruf', 'anies baswedan': 'anies',
    'prabowo subianto': 'prabowo', 'sandiaga uno': 'sandiaga', 'ridwan kamil': 'rk',
    'ganjar pranowo': 'ganjar', 'airlangga hartarto': 'airlangga', 'luhut pandjaitan': 'luhut',
    'mahfud md': 'mahfud', 'muhadjir effendy': 'muhadjir', 'ahmad riza patria': 'riza patria',
    'gibran rakabuming': 'gibran', 'bima arya sugiarto': 'bima arya', 'wiku adisasmito': 'wiku',
    'achmad yurianto': 'yurianto', 'agus harimurti': 'ahy', 'agus harimurti yudhoyono': 'ahy',
    'susilo bambang yudhoyono': 'sby', 'puan maharani': 'puan', 'donald trump': 'trump',
    'joe biden': 'biden', 'vladimir putin': 'putin'
}
PUBLIC_FIGURES = list(set(list(CANONICAL_FIGURES.keys()) + list(CANONICAL_FIGURES.values()) + [
    'terawan', 'budi gunadi', 'erick thohir', 'sri mulyani', 'dedie rachim',
    'bobby nasution', 'doni monardo', 'idham azis', 'listyo sigit', 'hadi tjahjanto',
    'nadiem', 'nadiem makarim', 'megawati', 'rizieq', 'rizieq shihab',
    'sinovac', 'pfizer', 'moderna', 'astrazeneca', 'novavax', 'bio farma'
]))
FIGURE_REGEX = re.compile(r'\b(' + '|'.join(re.escape(f) for f in sorted(PUBLIC_FIGURES, key=len, reverse=True)) + r')\b', re.I)

# 6. Lembaga, Institusi, & Penanda Entitas Komposit
INSTITUTIONS = {
    'itb', 'ugm', 'ui', 'unpad', 'undip', 'ipb', 'unair', 'its', 'unhas', 'unsri', 'usu',
    'uny', 'stie', 'ekuitas', 'kemenkes', 'menristek', 'bnpb', 'satgas', 'kpu', 'gerindra',
    'dpc', 'dprd', 'dpr', 'polri', 'bareskrim', 'dpmptsp', 'menko', 'pmk', 'polda', 'polres',
    'kpk', 'bpom', 'who', 'mui', 'kemenhub', 'kemendagri', 'kemensos', 'kemendikbud'
}

INSTITUTION_MARKERS = [
    'bea cukai', 'kantor bea cukai', 'rsud', 'puskesmas', 'dinkes', 'dinas kesehatan',
    'pemprov', 'pemkot', 'pemkab', 'polres', 'polda', 'kodim', 'pasar',
    'bupati', 'walikota', 'gubernur', 'kapolres', 'dprd', 'sekolah'
]
re_inst_marker = re.compile(r'\b(' + '|'.join(re.escape(m) for m in sorted(INSTITUTION_MARKERS, key=len, reverse=True)) + r')\b', re.I)

# 7. Kata Negasi & Verba Negatif Inheren
NEGATION_WORDS = {
    'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak',
    'bebas', 'nihil', 'nol', 'stop', 'hentikan', 'anti', 'menolak', 'membantah',
    'menyanggah', 'menafikan', 'mustahil'
}
NEG_WORDS = {'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak', 'anti', 'mustahil'}

INHERENT_NEG_VERBS = {
    'larang', 'melarang', 'dilarang', 'batal', 'membatalkan', 'dibatalkan',
    'gagal', 'menggagalkan', 'digagalkan', 'tolak', 'menolak', 'ditolak',
    'bantah', 'membantah', 'dibantah', 'tunda', 'menunda', 'ditunda'
}

PREFIXES = ['meng', 'meny', 'men', 'mem', 'me', 'di', 'ber', 'ter', 'pen', 'pem', 'peny', 'pe', 'ke', 'se']
SUFFIXES = ['kan', 'an', 'i', 'nya', 'lah', 'kah']

def light_stem(word):
    if len(word) <= 4:
        return word
    w = word
    for pref in PREFIXES:
        if w.startswith(pref) and len(w) - len(pref) >= 3:
            w = w[len(pref):]
            break
    for suff in SUFFIXES:
        if w.endswith(suff) and len(w) - len(suff) >= 3:
            w = w[:-len(suff)]
            break
    return w

def get_canonical_locations(text):
    raw_locs = [m.group(0).lower() for m in LOC_REGEX.finditer(text)]
    return set(CANONICAL_LOC.get(loc, loc) for loc in raw_locs)

def get_canonical_figures(text):
    raw_figs = [m.group(0).lower() for m in FIGURE_REGEX.finditer(text)]
    return set(CANONICAL_FIGURES.get(f, f) for f in raw_figs)

def extract_numbers_comprehensive(text):
    norm = re.sub(r'(\d+)\s+(\d+)', r'\1\2', text)
    num_matches = set(re.findall(r'\b\d+(?:[.,]\d+)?%?\b', norm))
    return {n for n in num_matches if not (len(n) == 4 and n.isdigit() and n.startswith(('19', '20')))}

def clean_title(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

print("Modul leksikal kanonikal, ekspansi antonim, leksikon sanggahan, parser angka, dan skop negasi/entitas komposit siap!")

Modul leksikal kanonikal, ekspansi antonim, leksikon sanggahan, parser angka, dan skop negasi/entitas komposit siap!


## 3. Ekstraksi Fitur Leksikal & Okapi BM25 (*Strict Train-Only Fitting*)
Sesuai **Rule #8 (No Data Leakage)**, seluruh vectorizer TF-IDF dan Okapi BM25 **hanya di-fit pada data train** (`train.csv`), kemudian ditransformasikan ke data train dan test.

In [4]:
from sklearn.decomposition import TruncatedSVD

if 'clean_title' not in globals():
    def clean_title(text):
        return re.sub(r'\s+', ' ', str(text)).strip()

train_titles = [clean_title(t) for t in train['title'].fillna('').astype(str).tolist()]
train_contents = train['content'].fillna('').astype(str).tolist()
train_fs = [" ".join(c.split()[:40]) for c in train_contents]

test_titles = [clean_title(t) for t in test['title'].fillna('').astype(str).tolist()]
test_contents = test['content'].fillna('').astype(str).tolist()
test_fs = [" ".join(c.split()[:40]) for c in test_contents]

# TF-IDF Vectorization
print("Fitting TF-IDF Word & Char Vectorizers (Strict Train-Only Fitting)...")
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15000, sublinear_tf=True)
tfidf_word.fit(train_titles + train_contents)

tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=15000, sublinear_tf=True)
tfidf_char.fit(train_titles + train_contents)

# Latent Semantic Analysis (TruncatedSVD 100 Components - Rule #4 Compliant)
print("Fitting TruncatedSVD LSA (100 components)...")
svd = TruncatedSVD(n_components=100, random_state=SEED)
train_c_w = tfidf_word.transform(train_contents)
svd.fit(train_c_w)

def compute_corpus_stats(tfidf_vec, corpus_texts):
    X = tfidf_vec.transform(corpus_texts)
    doc_lens = np.asarray(X.sum(axis=1)).flatten()
    avg_dl = float(np.mean(doc_lens))
    idfs = tfidf_vec.idf_
    return avg_dl, idfs

print("Computing BM25 & Pairwise Similarity Matrices...")
avg_dl_c, idf_w = compute_corpus_stats(tfidf_word, train_contents)
avg_dl_fs, _ = compute_corpus_stats(tfidf_word, train_fs)

def calc_bm25_score(t_vec, c_vec, c_len, avg_dl, k1=1.5, b=0.75):
    c_indices = c_vec.indices
    c_data = c_vec.data
    t_indices = t_vec.indices
    t_data = t_vec.data
    if len(t_indices) == 0 or len(c_indices) == 0:
        return 0.0
    c_dict = dict(zip(c_indices, c_data))
    t_dict = dict(zip(t_indices, t_data))
    common = set(t_indices).intersection(c_indices)
    if not common:
        return 0.0
    score = 0.0
    len_norm = 1.0 - b + b * (c_len / (avg_dl + 1e-9))
    for idx in common:
        tf_c = c_dict[idx]
        tf_t = t_dict[idx]
        idf_val = idf_w[idx] if idx < len(idf_w) else 1.0
        term_score = idf_val * (tf_c * (k1 + 1.0)) / (tf_c + k1 * len_norm) * tf_t
        score += term_score
    return float(score)

def get_pairwise_vectors(titles, contents, fs_texts):
    t_w = tfidf_word.transform(titles)
    c_w = tfidf_word.transform(contents)
    fs_w = tfidf_word.transform(fs_texts)
    
    t_c = tfidf_char.transform(titles)
    c_c = tfidf_char.transform(contents)
    fs_c = tfidf_char.transform(fs_texts)
    
    cos_w_c = np.asarray((t_w.multiply(c_w)).sum(axis=1)).flatten()
    cos_w_fs = np.asarray((t_w.multiply(fs_w)).sum(axis=1)).flatten()
    cos_c_c = np.asarray((t_c.multiply(c_c)).sum(axis=1)).flatten()
    cos_c_fs = np.asarray((t_c.multiply(fs_c)).sum(axis=1)).flatten()
    
    t_lsa = svd.transform(t_w)
    c_lsa = svd.transform(c_w)
    fs_lsa = svd.transform(fs_w)
    
    cos_lsa_c = (t_lsa * c_lsa).sum(axis=1) / (np.linalg.norm(t_lsa, axis=1) * np.linalg.norm(c_lsa, axis=1) + 1e-9)
    cos_lsa_fs = (t_lsa * fs_lsa).sum(axis=1) / (np.linalg.norm(t_lsa, axis=1) * np.linalg.norm(fs_lsa, axis=1) + 1e-9)
    
    c_lens = np.asarray(c_w.sum(axis=1)).flatten()
    fs_lens = np.asarray(fs_w.sum(axis=1)).flatten()
    bm25_c = [calc_bm25_score(t_w[i], c_w[i], c_lens[i], avg_dl_c) for i in range(len(titles))]
    bm25_fs = [calc_bm25_score(t_w[i], fs_w[i], fs_lens[i], avg_dl_fs) for i in range(len(titles))]
    
    return cos_w_c, cos_w_fs, cos_c_c, cos_c_fs, np.array(bm25_c), np.array(bm25_fs), cos_lsa_c, cos_lsa_fs, t_w

cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr, bm25_c_tr, bm25_fs_tr, cos_lsa_c_tr, cos_lsa_fs_tr, tr_t_w = get_pairwise_vectors(train_titles, train_contents, train_fs)
cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te, bm25_c_te, bm25_fs_te, cos_lsa_c_te, cos_lsa_fs_te, te_t_w = get_pairwise_vectors(test_titles, test_contents, test_fs)

print("TF-IDF, Okapi BM25, dan TruncatedSVD LSA (100 komponen) berhasil di-fit secara ketat HANYA pada data train!")

Fitting TF-IDF Word & Char Vectorizers (Strict Train-Only Fitting)...
Fitting TruncatedSVD LSA (100 components)...
Computing BM25 & Pairwise Similarity Matrices...
TF-IDF, Okapi BM25, dan TruncatedSVD LSA (100 komponen) berhasil di-fit secara ketat HANYA pada data train!


## 4. Pipeline Rekayasa 49 Fitur Teroptimasi (Sentence-Level Semantic Alignment, Inverted Pyramid Ratio, Institutional & City Conflict, High-IDF Absence)

Pipeline ekstraksi fitur menggabungkan sinyal leksikal, semantik, wacana teks, perbandingan entitas kanonikal, 4 Dynamic Open-NER, 3 Fitur Country/Bigram, 3 Fitur Terobosan (Pure Lead Loc/Fig & Security Rank), serta **6 Fitur Semantik & Diskursus Tingkat Lanjut** secara **100% murni dan independen per-baris (*row-independent*)**:

Berdasarkan investigasi diagnostik *False Negatives*:
1. **Sentence-Level Discourse Alignment & Inverted Pyramid Ratio**: Fitur `feat_lead_sent_cos`, `feat_max_sent_cos`, dan `feat_title_lead_ratio` mengevaluasi keselarasan topik judul secara spesifik pada kalimat pembuka (*lead sentence*) vs tubuh artikel, memastikan berita mematuhi struktur piramida terbalik jurnalistik.
2. **Distinctive High-IDF Keyword Absence**: Fitur `feat_high_idf_absent` menghitung rasio kata kunci judul berbobot IDF tinggi ($\ge 5.0$) yang sama sekali tidak muncul di tubuh berita.
3. **Institutional & Security Apparatus Conflict**: Fitur `feat_inst_conf` mendeteksi pertukaran lembaga negara/keamanan (Polri vs TNI, KPU vs Bawaslu, KPK vs Kejaksaan, nomor komisi DPR) dengan kemurnian hoaks mencapai **73.91%**.
4. **City-Level Lead Conflict**: Fitur `feat_city_lead_conf` membandingkan entitas kota pada pembuka artikel tanpa terhalang kesamaan nama provinsi (*province masking*), menangkap hoaks dengan presisi **73.08%**.

In [5]:
def extract_features(df_titles, df_contents, df_fs,
                     cos_wc, cos_wfs, cos_cc, cos_cfs,
                     bm25_c, bm25_fs,
                     cos_lsa_c, cos_lsa_fs,
                     title_tfidf_matrix):
    vocab_word = tfidf_word.vocabulary_
    idfs_word = tfidf_word.idf_
    feats = []
    best_win_texts = []
    N_items = len(df_titles)
    
    for i in range(N_items):
        t_raw = df_titles[i]
        c_raw = df_contents[i]
        t_lower = t_raw.lower()
        c_lower = c_raw.lower()
        fs_lower = df_fs[i].lower()
        
        t_words = t_raw.split()
        c_words = c_raw.split()
        t_tokens = set(re.findall(r'\w+', t_lower))
        c_tokens = set(re.findall(r'\w+', c_lower))
        fs_tokens = set(re.findall(r'\w+', fs_lower))
        
        useful_t = [w for w in t_tokens if w not in STOP_WORDS and len(w) > 2]
        
        # 1. Canonical Locations
        t_locs = get_canonical_locations(t_lower)
        c_locs = get_canonical_locations(c_lower)
        fs_locs = get_canonical_locations(fs_lower)
        loc_missing = sum(1 for loc in t_locs if loc not in c_locs and not any(p in c_tokens for p in loc.split()))
        loc_mismatch = loc_missing / (len(t_locs) + 1e-9) if t_locs else 0.0
        loc_conflict = 1.0 if (t_locs and fs_locs and loc_missing > 0 and not t_locs.intersection(fs_locs)) else 0.0
        loc_c_conflict = 1.0 if (t_locs and c_locs and loc_missing > 0 and not t_locs.intersection(c_locs)) else 0.0
        
        # 2. Canonical Figures
        t_figs = get_canonical_figures(t_lower)
        c_figs = get_canonical_figures(c_lower)
        fig_missing = sum(1 for fig in t_figs if fig not in c_figs and not any(p in c_tokens for p in fig.split()))
        fig_mismatch = fig_missing / (len(t_figs) + 1e-9) if t_figs else 0.0
        fig_conflict = 1.0 if (t_figs and c_figs and fig_missing > 0 and not t_figs.intersection(c_figs)) else 0.0
        
        # 3. Institutions
        t_inst = {w for w in t_tokens if w in INSTITUTIONS}
        inst_missing = len([inst for inst in t_inst if inst not in c_tokens])
        inst_mismatch = inst_missing / (len(t_inst) + 1e-9) if t_inst else 0.0
        
        # 4. Token & Stemming Coverage
        exact_match = sum(1 for w in useful_t if w in c_tokens)
        c_stems = set(light_stem(w) for w in c_tokens if len(w) > 2)
        stem_match = sum(1 for w in useful_t if (w in c_tokens or light_stem(w) in c_stems))
        tok_cov_exact = exact_match / (len(useful_t) + 1e-9) if useful_t else 1.0
        tok_cov_stem = stem_match / (len(useful_t) + 1e-9) if useful_t else 1.0
        stem_gain = tok_cov_stem - tok_cov_exact
        
        # 5. Multi-Scale Discourse Windows (25, 50, 80 token)
        lead_25_tokens = set(re.findall(r'\w+', " ".join(c_words[:25]).lower()))
        lead_50_tokens = set(re.findall(r'\w+', " ".join(c_words[:50]).lower()))
        lead_80_tokens = set(re.findall(r'\w+', " ".join(c_words[:80]).lower()))
        
        lead_25_cov = len([w for w in useful_t if w in lead_25_tokens]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        lead_50_cov = len([w for w in useful_t if w in lead_50_tokens]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        lead_80_cov = len([w for w in useful_t if w in lead_80_tokens]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        
        windows_50 = []
        w_size = 50
        step = 20
        for s in range(0, max(1, len(c_words)), step):
            chunk = " ".join(c_words[s:s+w_size])
            if len(chunk.split()) >= 5:
                windows_50.append(chunk)
        if not windows_50:
            windows_50 = [c_raw]
            
        best_overlap_50 = 0
        best_win_idx_50 = 0
        overlap_counts = []
        for w_idx, win in enumerate(windows_50):
            wtoks = set(re.findall(r'\w+', win.lower()))
            overlap = len([w for w in useful_t if w in wtoks])
            overlap_counts.append(overlap)
            if overlap > best_overlap_50:
                best_overlap_50 = overlap
                best_win_idx_50 = w_idx
                
        best_50_cov = best_overlap_50 / (len(useful_t) + 1e-9) if useful_t else 1.0
        best_win_pos_norm = best_win_idx_50 / (len(windows_50) + 1e-9)
        best_minus_lead = best_50_cov - lead_50_cov
        best_win_texts.append(windows_50[best_win_idx_50])
        
        sorted_overlaps = sorted(overlap_counts, reverse=True)
        top1_overlap = sorted_overlaps[0] if sorted_overlaps else 0
        top2_overlap = sorted_overlaps[1] if len(sorted_overlaps) > 1 else top1_overlap
        top1_top2_contrast = (top1_overlap - top2_overlap) / (len(useful_t) + 1e-9) if useful_t else 0.0
        
        # 6. Targeted Fact Guards
        head_ent = ""
        for j, w in enumerate(t_words[:4]):
            clean_w = re.sub(r'\W+', '', w)
            if clean_w and clean_w.lower() not in STOP_WORDS and len(clean_w) > 2:
                if w[0].isupper():
                    if j+1 < len(t_words) and t_words[j+1][0].isupper():
                        next_w = re.sub(r'\W+', '', t_words[j+1])
                        head_ent = f"{clean_w.lower()} {next_w.lower()}"
                    else:
                        head_ent = clean_w.lower()
                break
        head_entity_absence = 1.0 if (head_ent and head_ent not in c_lower) else 0.0
        
        num_units = re_num_unit.findall(t_lower)
        if num_units:
            matches = 0
            for num, unit in num_units:
                pat = rf"{re.escape(num)}\s+(?:\w+\s+){{0,2}}{re.escape(unit)}"
                if re.search(pat, c_lower):
                    matches += 1
            num_unit_match_ratio = matches / len(num_units)
        else:
            num_unit_match_ratio = 1.0
            
        t_negs = t_tokens & NEGATION_WORDS
        best_w_tokens = set(re.findall(r'\w+', windows_50[best_win_idx_50].lower()))
        win_negs = best_w_tokens & NEGATION_WORDS
        lead_negs = lead_50_tokens & NEGATION_WORDS
        neg_inversion = 1.0 if (bool(t_negs) != bool(win_negs)) else 0.0
        lead_neg_asymmetry = 1.0 if (bool(t_negs) != bool(lead_negs)) else 0.0
        
        refuting_matches = len(re_refuting.findall(windows_50[best_win_idx_50]))
        refuting_score = float(min(refuting_matches, 3))
        
        t_clean_words = [w for w in re.findall(r'\b[a-z0-9]+\b', t_lower) if len(w) > 2 and w not in STOP_WORDS]
        if len(t_clean_words) >= 2:
            title_ordered_bgs = [f"{t_clean_words[j]} {t_clean_words[j+1]}" for j in range(len(t_clean_words)-1)]
            matched_bgs = sum(1 for bg in title_ordered_bgs if bg in c_lower)
            ordered_bigram_ratio = matched_bgs / len(title_ordered_bgs)
        else:
            ordered_bigram_ratio = 1.0
            
        # 7. Antonym & Conflict
        ada_conflict = 0.0
        if ('ada' in t_tokens and not t_negs):
            if 'tidak ada' in fs_lower or 'tak ada' in fs_lower:
                ada_conflict = 1.0
        elif ('tidak ada' in t_lower or 'tak ada' in t_lower):
            if 'ada' in fs_tokens and 'tidak ada' not in fs_lower and 'tak ada' not in fs_lower:
                ada_conflict = 1.0
                
        antonym_conflict = 0.0
        for w1, w2 in EXPANDED_ANTONYM_PAIRS:
            if (w1 in t_tokens and w2 in c_tokens) or (w2 in t_tokens and w1 in c_tokens):
                antonym_conflict = 1.0
                break
                
        # 8. Numbers Comprehensive
        t_subst_nums = extract_numbers_comprehensive(t_raw)
        c_subst_nums = extract_numbers_comprehensive(c_raw)
        num_missing_subst = len(t_subst_nums - c_subst_nums)
        num_cov_subst = len(t_subst_nums.intersection(c_subst_nums)) / (len(t_subst_nums) + 1e-9) if t_subst_nums else 1.0
        
        # 9. Capital Words & Lengths
        is_title_case = sum(1 for w in t_words if w and w[0].isupper()) > 0.8 * len(t_words)
        if not is_title_case:
            cap_words = [w for w in t_words if len(w) > 1 and w[0].isupper() and w.lower() not in STOP_WORDS]
            cap_missing = sum(1 for w in cap_words if w.lower() not in c_tokens)
            cap_missing_fs = sum(1 for w in cap_words if w.lower() not in fs_tokens)
            cap_cov_c = 1.0 - (cap_missing / (len(cap_words) + 1e-9)) if cap_words else 1.0
            cap_cov_fs = 1.0 - (cap_missing_fs / (len(cap_words) + 1e-9)) if cap_words else 1.0
        else:
            cap_cov_c = 1.0
            cap_cov_fs = 1.0
            
        head_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS][:3]
        head_missing_c = sum(1 for hw in head_content_words if hw not in c_tokens)
        head_cov_c = 1.0 - (head_missing_c / (len(head_content_words) + 1e-9)) if head_content_words else 1.0
        
        len_ratio = len(c_raw) / (len(t_raw) + 1e-9)
        
        # 10. Adaptive Discourse Inversion Scopes (4 Fitur Baru Penekan False Negatives)
        t_lower_words = t_lower.split()
        lead_words = c_lower.split()[:60]
        lead_text = " ".join(lead_words)
        
        # 10a. Verb Negation Scope Conflict
        v_conf = 0.0
        for idx, w in enumerate(t_lower_words):
            is_action = (w in INHERENT_NEG_VERBS) or (len(w) > 4 and w.startswith(('me', 'di', 'ber', 'ter'))) or w in ['bisa', 'dapat', 'sebabkan', 'izinkan', 'bolehkan']
            if is_action:
                t_is_neg = (w in INHERENT_NEG_VERBS) or (idx > 0 and t_lower_words[idx-1] in NEG_WORDS) or (idx > 1 and t_lower_words[idx-2] in NEG_WORDS)
                c_instances = []
                for j, cw in enumerate(lead_words):
                    if cw == w or (len(w) >= 5 and len(cw) >= 5 and w[:5] == cw[:5]):
                        c_neg = (cw in INHERENT_NEG_VERBS) or (j > 0 and lead_words[j-1] in NEG_WORDS) or (j > 1 and lead_words[j-2] in NEG_WORDS)
                        c_instances.append(c_neg)
                if c_instances:
                    if not t_is_neg and all(c_instances):
                        v_conf = 1.0
                        break
                    elif t_is_neg and not any(c_instances):
                        v_conf = 1.0
                        break
                        
        # 10b. Existential Polarity Conflict
        e_conf = 0.0
        t_has_ada = 'ada ' in t_lower[:25] or t_lower.startswith('ada ')
        t_neg_ada = 'tidak ada' in t_lower[:30] or 'tak ada' in t_lower[:30]
        c_has_ada = 'ada ' in lead_text[:30] or lead_text.startswith('ada ')
        c_neg_ada = 'tidak ada' in lead_text[:35] or 'tak ada' in lead_text[:35]
        if (t_has_ada and not t_neg_ada) and c_neg_ada:
            e_conf = 1.0
        elif t_neg_ada and (c_has_ada and not c_neg_ada):
            e_conf = 1.0
            
        # 10c. Composite Entity Conflict
        comp_conf = 0.0
        m_inst = list(re_inst_marker.finditer(t_lower))
        m_loc_t = list(LOC_REGEX.finditer(t_lower))
        if m_inst and m_loc_t:
            for inst in m_inst:
                for loc in m_loc_t:
                    if 0 <= loc.start() - inst.end() <= 30 or 0 <= inst.start() - loc.end() <= 30:
                        inst_name = inst.group(0)
                        loc_name_t = loc.group(0)
                        idx_c = c_lower.find(inst_name)
                        if idx_c != -1:
                            surrounding = c_lower[max(0, idx_c-30):min(len(c_lower), idx_c+len(inst_name)+30)]
                            locs_c = [m.group(0) for m in LOC_REGEX.finditer(surrounding)]
                            if locs_c and loc_name_t not in locs_c:
                                comp_conf = 1.0
                                break
                if comp_conf == 1.0:
                    break
                    
        # 10d. Directional Role Inversion
        dir_conf = 0.0
        prep_m = list(re.finditer(r'\b(ke|kepada)\s+([a-zA-Z]+)', t_lower))
        if prep_m:
            for m in prep_m:
                target = m.group(2)
                if len(target) > 3 and target not in ['mana', 'sana', 'depan', 'dalam', 'luar']:
                    actor_pat = rf"\b{re.escape(target)}\s+(?:akan|telah|sudah|kembali|bakal|siap|resmi|mulai|menyumbang|menyerahkan|memberi)"
                    if re.search(actor_pat, lead_text):
                        dir_conf = 1.0
                        break
        
        # 11. Fine-Grained Entity Alignment & Sentence Matching (Fitur 46, 47, 48)
        # 11a & 11b. Bobot IDF Kata Judul yang Hilang (Unmatched High-IDF Title Words)
        t_idfs = [idfs_word[vocab_word[w]] for w in t_tokens if w in vocab_word]
        if t_idfs:
            unm_words = [w for w in t_tokens if w not in c_tokens and w in vocab_word]
            unm_idfs = [idfs_word[vocab_word[w]] for w in unm_words]
            max_unm_idf = max(unm_idfs) if unm_idfs else 0.0
            sum_unm_ratio = sum(unm_idfs) / (sum(t_idfs) + 1e-9)
        else:
            max_unm_idf = 0.0
            sum_unm_ratio = 0.0

        # 11c. Kemiripan Kosinus Maksimum Kalimat Tunggal (Max Single-Sentence Cosine)
        sents = [s.strip() for s in re.split(r'[.\n\r?!]+', c_raw) if len(s.strip().split()) >= 3]
        if sents:
            t_v = title_tfidf_matrix[i]
            s_v = tfidf_word.transform(sents)
            s_cos = np.asarray((t_v.multiply(s_v)).sum(axis=1)).flatten()
            max_sent_cos = float(np.max(s_cos)) if len(s_cos) > 0 else 0.0
        else:
            max_sent_cos = 0.0
        
        feats.append([
            cos_wc[i], cos_wfs[i], cos_cc[i], cos_cfs[i],
            bm25_c[i], bm25_fs[i],
            cos_lsa_c[i], cos_lsa_fs[i],
            best_50_cov, best_win_pos_norm, best_minus_lead,
            lead_25_cov, lead_50_cov, lead_80_cov, top1_top2_contrast,
            loc_mismatch, loc_conflict, loc_c_conflict,
            fig_mismatch, fig_conflict,
            inst_mismatch,
            tok_cov_exact, tok_cov_stem, stem_gain, head_cov_c,
            head_entity_absence, num_unit_match_ratio,
            neg_inversion, lead_neg_asymmetry, ada_conflict, antonym_conflict,
            refuting_score, ordered_bigram_ratio,
            num_missing_subst, num_cov_subst,
            cap_cov_c, cap_cov_fs, len_ratio, len(t_words), len(c_words),
            v_conf, e_conf, comp_conf, dir_conf,
            max_unm_idf, sum_unm_ratio, max_sent_cos
        ])
    
    # Ultra-fast Batch TF-IDF transform for best focal window cosine
    win_matrix = tfidf_word.transform(best_win_texts)
    best_win_tfidf_cos = np.asarray((title_tfidf_matrix.multiply(win_matrix)).sum(axis=1)).flatten()
    
    feature_matrix = np.array(feats)
    # Insert best_win_tfidf_cos at column index 8
    feature_matrix = np.insert(feature_matrix, 8, best_win_tfidf_cos, axis=1)
    return feature_matrix

print("Mengekstrak 48 fitur teroptimasi (Fine-Grained Alignment & Sentences) untuk data train dan test...")
X_train_raw = extract_features(
    train_titles, train_contents, train_fs,
    cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr,
    bm25_c_tr, bm25_fs_tr,
    cos_lsa_c_tr, cos_lsa_fs_tr,
    tr_t_w
)
X_test_raw = extract_features(
    test_titles, test_contents, test_fs,
    cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te,
    bm25_c_te, bm25_fs_te,
    cos_lsa_c_te, cos_lsa_fs_te,
    te_t_w
)
y_train = train['label'].values

# Pemilihan 33 Fitur Bersih (Pruning 15 Fitur Noisy/Redundan)
# Membuang fitur-fitur yang berkorelasi negatif / redundan (seperti antonym_conflict, len_ratio, bm25_fs, dll.)
# untuk memangkas False Positive secara drastis dan memaksimalkan presisi kelas minoritas.
DROP_FEATURE_INDICES = [4, 5, 7, 11, 25, 26, 27, 31, 34, 35, 38, 39, 43, 46, 47]
KEEP_FEATURE_INDICES = [i for i in range(48) if i not in DROP_FEATURE_INDICES]

X_train_33 = X_train_raw[:, KEEP_FEATURE_INDICES]
X_test_33 = X_test_raw[:, KEEP_FEATURE_INDICES]

# =========================================================================
# Ekstraksi 4 Fitur Dynamic Open-NER & Subject Alignment (Chesney et al. 2017)
# =========================================================================
ROLE_PREFIXES = r'(?:Presiden|Wapres|Gubernur|Wagub|Bupati|Walikota|Kapolri|Kapolda|Kapolres|Panglima|Kasad|Kasal|Kasau|Menteri|Menko|Menkes|Menkeu|Mendagri|Menlu|Menpora|Menaker|Dirjen|Dirut|KPK|Jaksa|Hakim|Ketua|Wakil|Dokter|dr|drg|Spesialis|Prof|Ustadz|Habib|Kiai|Pendeta|Penyanyi|Aktor|Aktris|Musisi|Vokalis|Pelatih|Coach|Pebalap|Kiper|Striker|Kapten|Bintang|Artis)\b'
ROLE_NAME_REGEX = re.compile(rf'{ROLE_PREFIXES}\s+([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*)', re.I)

FACILITY_PREFIXES = r'(?:Rumah Sakit|RSUD|RS|Puskesmas|Dinas|Kementerian|Kemendikbud|Kemenkes|Kemenkeu|Kemendagri|Polres|Polda|Polsek|Kodim|Koramil|Pengadilan|Kejaksaan|Bandara|Stasiun|Pelabuhan|Terminal|Stadion|Gelora|Pasar|Mal|Mall|Plaza|Hotel|Apartemen|Universitas|Institut|Fakultas|Sekolah|SMA|SMP|SD|PT|CV)\b'
FACILITY_NAME_REGEX = re.compile(rf'{FACILITY_PREFIXES}\s+([A-Z][a-zA-Z0-9]+(?:\s+[A-Z][a-zA-Z0-9]+)*)', re.I)

ACRONYM_REGEX = re.compile(r'\b[A-Z]{3,8}\b')

def extract_dynamic_ner_features(titles, contents):
    N = len(titles)
    feat_role = np.zeros((N, 1), dtype=np.float32)
    feat_fac = np.zeros((N, 1), dtype=np.float32)
    feat_acro = np.zeros((N, 1), dtype=np.float32)
    feat_head = np.zeros((N, 1), dtype=np.float32)
    
    for i in range(N):
        t_raw, c_raw = titles[i], contents[i]
        c_lower = c_raw.lower()
        
        # 1. Gelar/Jabatan + Nama Tokoh yang hilang di isi berita
        m_r = ROLE_NAME_REGEX.findall(t_raw)
        if m_r and not any(r.lower() in c_lower for r in m_r):
            feat_role[i, 0] = 1.0
            
        # 2. Fasilitas / Instansi Publik yang hilang di isi berita
        m_f = FACILITY_NAME_REGEX.findall(t_raw)
        if m_f and not any(f.lower() in c_lower for f in m_f):
            feat_fac[i, 0] = 1.0
            
        # 3. Akronim Organisasi/Istilah yang hilang di isi berita
        m_a = ACRONYM_REGEX.findall(t_raw)
        if m_a and not any(a.lower() in c_lower for a in m_a):
            feat_acro[i, 0] = 1.0
            
        # 4. Kata Pokok Subjek Judul (Head Subject) yang hilang di isi berita
        t_words = [w for w in re.findall(r'[a-z0-9]+', t_raw.lower()) if w not in STOP_WORDS]
        if t_words and t_words[0] not in c_lower:
            feat_head[i, 0] = 1.0
            
    return np.hstack([feat_role, feat_fac, feat_acro, feat_head])

ner_train = extract_dynamic_ner_features(train_titles, train_contents)
ner_test = extract_dynamic_ner_features(test_titles, test_contents)

# =========================================================================
# Ekstraksi Fitur Entitas Negara & Lead Bigrams (Country Conflict & Inverted Pyramid)
# =========================================================================
COUNTRIES_MAP = {
    'amerika serikat': 'amerika', 'amerika': 'amerika', 'usa': 'amerika',
    'brasil': 'brasil', 'brazil': 'brasil',
    'china': 'china', 'tiongkok': 'china',
    'india': 'india', 'inggris': 'inggris', 'britania': 'inggris',
    'italia': 'italia', 'italy': 'italia',
    'jepang': 'jepang', 'japan': 'jepang',
    'korea selatan': 'korsel', 'korsel': 'korsel',
    'korea utara': 'korut', 'korut': 'korut',
    'malaysia': 'malaysia', 'singapura': 'singapura', 'singapore': 'singapura',
    'rusia': 'rusia', 'russia': 'rusia',
    'prancis': 'prancis', 'perancis': 'prancis',
    'jerman': 'jerman', 'germany': 'jerman',
    'australia': 'australia', 'spanyol': 'spanyol', 'spain': 'spanyol',
    'filipina': 'filipina', 'thailand': 'thailand', 'vietnam': 'vietnam',
    'arab saudi': 'arab saudi', 'saudi': 'arab saudi',
    'iran': 'iran', 'turki': 'turki', 'turkey': 'turki',
    'meksiko': 'meksiko', 'afrika selatan': 'afsel',
    'kanada': 'kanada', 'belanda': 'belanda', 'swiss': 'swiss', 'mesir': 'mesir'
}
COUNTRIES_PAT = re.compile(r'\b(?:' + '|'.join(re.escape(k) for k in COUNTRIES_MAP.keys()) + r')\b', re.I)

def extract_country_and_lead_features(titles, contents):
    N_items = len(titles)
    feat_ctry_mismatch = np.zeros((N_items, 1), dtype=np.float32)
    feat_ctry_conflict = np.zeros((N_items, 1), dtype=np.float32)
    feat_lead_bg = np.zeros((N_items, 1), dtype=np.float32)
    
    for i in range(N_items):
        t_low = titles[i].lower()
        c_low = contents[i].lower()
        lead_low = ' '.join(c_low.split()[:45])
        
        # 1. Country Analysis
        t_raw_c = COUNTRIES_PAT.findall(t_low)
        if t_raw_c:
            t_c = set(COUNTRIES_MAP[c] for c in t_raw_c)
            c_raw_c = COUNTRIES_PAT.findall(c_low)
            c_c = set(COUNTRIES_MAP[c] for c in c_raw_c)
            lead_raw_c = COUNTRIES_PAT.findall(lead_low)
            lead_c = set(COUNTRIES_MAP[c] for c in lead_raw_c)
            
            missing = [c for c in t_c if c not in c_c]
            if missing:
                feat_ctry_mismatch[i, 0] = len(missing) / len(t_c)
            if missing or (lead_c and not t_c.intersection(lead_c)):
                feat_ctry_conflict[i, 0] = 1.0
                
        # 2. Lead Bigram Match Ratio
        t_words = [w for w in re.findall(r'\b[a-z0-9]+\b', t_low) if len(w) > 2 and w not in STOP_WORDS]
        if len(t_words) >= 2:
            bgs = [f'{t_words[j]} {t_words[j+1]}' for j in range(len(t_words)-1)]
            matched = sum(1 for bg in bgs if bg in lead_low)
            feat_lead_bg[i, 0] = matched / len(bgs)
        else:
            feat_lead_bg[i, 0] = 1.0
            
    return feat_ctry_mismatch, feat_ctry_conflict, feat_lead_bg

ctry_mis_tr, ctry_conf_tr, lead_bg_tr = extract_country_and_lead_features(train_titles, train_contents)
ctry_mis_te, ctry_conf_te, lead_bg_te = extract_country_and_lead_features(test_titles, test_contents)

# =========================================================================
# Ekstraksi Fitur Terobosan: Pure Lead Entity Conflicts & Security Ranks
# =========================================================================
RANK_PREFIXES = r'(?:Irjen|Kapolda|Kombes|AKBP|Kompol|AKP|Iptu|Ipda|Aiptu|Aipda|Bripka|Brigadir|Briptu|Bripda|Jenderal|Letjen|Mayjen|Brigjen|Kolonel|Letkol|Mayor|Kapten|Lettu|Letda)\b'
RANK_REGEX = re.compile(rf'{RANK_PREFIXES}\s+([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*)', re.I)

def extract_pure_lead_conflicts(titles, contents):
    N_items = len(titles)
    feat_pure_loc = np.zeros((N_items, 1), dtype=np.float32)
    feat_pure_fig = np.zeros((N_items, 1), dtype=np.float32)
    feat_rank = np.zeros((N_items, 1), dtype=np.float32)
    
    for i in range(N_items):
        t_low = titles[i].lower()
        c_low = contents[i].lower()
        lead_low = ' '.join(c_low.split()[:40])
        
        # 1. Pure Lead Location Conflict
        t_loc = get_canonical_locations(t_low)
        lead_loc = get_canonical_locations(lead_low)
        if t_loc and lead_loc and not t_loc.intersection(lead_loc):
            feat_pure_loc[i, 0] = 1.0
            
        # 2. Pure Lead Figure Conflict
        t_fig = get_canonical_figures(t_low)
        lead_fig = get_canonical_figures(lead_low)
        if t_fig and lead_fig and not t_fig.intersection(lead_fig):
            feat_pure_fig[i, 0] = 1.0
            
        # 3. Security Rank Absence
        m_rk = RANK_REGEX.findall(titles[i])
        if m_rk and not any(r.lower() in c_low for r in m_rk):
            feat_rank[i, 0] = 1.0
            
    return feat_pure_loc, feat_pure_fig, feat_rank

pure_loc_tr, pure_fig_tr, rank_tr = extract_pure_lead_conflicts(train_titles, train_contents)
pure_loc_te, pure_fig_te, rank_te = extract_pure_lead_conflicts(test_titles, test_contents)

# =========================================================================
# Ekstraksi 6 Fitur Semantik & Diskursus Baru: Institutional, City-Level,
# Lead/Max Sentence Cosines, Inverted Pyramid Ratio, & High-IDF Absence
# =========================================================================
MUTUAL_INST = [
    ({'polri', 'polisi'}, {'tni', 'tentara', 'militer'}),
    ({'kpu'}, {'bawaslu'}),
    ({'kpk'}, {'kejagung', 'kejaksaan'}),
    ({'dpr'}, {'dpd'}),
]
KOMISI_PAT = re.compile(r'\bkomisi\s+(i|ii|iii|iv|v|vi|vii|viii|ix|x|xi)\b', re.I)

PROVINCES_SET = {
    'dki jakarta', 'jawa barat', 'jawa tengah', 'jawa timur', 'banten', 'yogyakarta', 'diy',
    'sumatera utara', 'sumatera barat', 'sumatera selatan', 'aceh', 'riau', 'kepulauan riau',
    'lampung', 'jambi', 'bengkulu', 'bangka belitung',
    'kalimantan timur', 'kalimantan barat', 'kalimantan selatan', 'kalimantan tengah', 'kalimantan utara',
    'sulawesi selatan', 'sulawesi utara', 'sulawesi tengah', 'sulawesi tenggara', 'sulawesi barat', 'gorontalo',
    'bali', 'nusa tenggara barat', 'nusa tenggara timur', 'maluku', 'maluku utara', 'papua', 'papua barat', 'indonesia'
}

vocab_word = tfidf_word.vocabulary_
idfs_word = tfidf_word.idf_

def extract_semantic_and_discourse_features(titles, contents):
    N_items = len(titles)
    feat_inst = np.zeros((N_items, 1), dtype=np.float32)
    feat_city = np.zeros((N_items, 1), dtype=np.float32)
    feat_lead_cos = np.zeros((N_items, 1), dtype=np.float32)
    feat_max_cos = np.zeros((N_items, 1), dtype=np.float32)
    feat_ratio = np.zeros((N_items, 1), dtype=np.float32)
    feat_idf_abs = np.zeros((N_items, 1), dtype=np.float32)
    
    t_matrix = tfidf_word.transform(titles)
    
    for i in range(N_items):
        t_low = titles[i].lower()
        c_low = contents[i].lower()
        lead_low = ' '.join(c_low.split()[:40])
        
        # 1. Institutional Conflict
        t_toks = set(re.findall(r'\b[a-z0-9]+\b', t_low))
        lead_toks = set(re.findall(r'\b[a-z0-9]+\b', lead_low))
        c_toks = set(re.findall(r'\b[a-z0-9]+\b', c_low))
        for setA, setB in MUTUAL_INST:
            t_has_A = bool(t_toks.intersection(setA))
            t_has_B = bool(t_toks.intersection(setB))
            lead_has_A = bool(lead_toks.intersection(setA))
            lead_has_B = bool(lead_toks.intersection(setB))
            if (t_has_A and not t_has_B and lead_has_B and not lead_has_A) or \
               (t_has_B and not t_has_A and lead_has_A and not lead_has_B):
                feat_inst[i, 0] = 1.0
                break
        if feat_inst[i, 0] == 0.0:
            m_t_kom = KOMISI_PAT.findall(t_low)
            if m_t_kom:
                m_lead_kom = KOMISI_PAT.findall(lead_low)
                m_c_kom = KOMISI_PAT.findall(c_low)
                t_num = m_t_kom[0].lower()
                if m_lead_kom and t_num not in [k.lower() for k in m_lead_kom]:
                    feat_inst[i, 0] = 1.0
                elif m_c_kom and t_num not in [k.lower() for k in m_c_kom]:
                    feat_inst[i, 0] = 1.0
                    
        # 2. City-Level Lead Conflict (Disentangled from Province Masking)
        t_locs = get_canonical_locations(t_low)
        lead_locs = get_canonical_locations(lead_low)
        t_cities = set(l for l in t_locs if l not in PROVINCES_SET)
        lead_cities = set(l for l in lead_locs if l not in PROVINCES_SET)
        if t_cities and lead_cities and not t_cities.intersection(lead_cities):
            feat_city[i, 0] = 1.0
            
        # 3, 4, 5. Sentence Cosines & Inverted Pyramid Alignment Ratio
        sents = [s.strip() for s in re.split(r'[.\n\r?!]+', contents[i]) if len(s.strip().split()) >= 3]
        if sents:
            s_v = tfidf_word.transform(sents)
            s_cos = np.asarray((t_matrix[i].multiply(s_v)).sum(axis=1)).flatten()
            max_c = float(np.max(s_cos)) if len(s_cos) > 0 else 0.0
            lead_c = float(s_cos[0]) if len(s_cos) > 0 else 0.0
            feat_lead_cos[i, 0] = lead_c
            feat_max_cos[i, 0] = max_c
            feat_ratio[i, 0] = lead_c / (max_c + 1e-6)
            
        # 6. High-IDF Absent Words Ratio
        high_idf_t = [w for w in set(re.findall(r'\b[a-z0-9]+\b', t_low)) if w in vocab_word and idfs_word[vocab_word[w]] >= 5.0]
        if high_idf_t:
            absent = [w for w in high_idf_t if w not in c_toks]
            feat_idf_abs[i, 0] = len(absent) / len(high_idf_t)
            
    return feat_inst, feat_city, feat_lead_cos, feat_max_cos, feat_ratio, feat_idf_abs

sem_tr = extract_semantic_and_discourse_features(train_titles, train_contents)
sem_te = extract_semantic_and_discourse_features(test_titles, test_contents)

# Gabungkan 33 Fitur Bersih + 4 Dynamic NER + 3 Fitur Country/Bigram + 3 Fitur Terobosan + 6 Fitur Semantik -> 49 Fitur Teroptimasi
X_train = np.hstack([X_train_33, ner_train, ctry_mis_tr, ctry_conf_tr, lead_bg_tr, pure_loc_tr, pure_fig_tr, rank_tr] + list(sem_tr))
X_test = np.hstack([X_test_33, ner_test, ctry_mis_te, ctry_conf_te, lead_bg_te, pure_loc_te, pure_fig_te, rank_te] + list(sem_te))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_raw shape: {X_train_raw.shape} -> X_train (49 Fitur Teroptimasi): {X_train.shape}")
print(f"X_test_raw shape : {X_test_raw.shape} -> X_test  (49 Fitur Teroptimasi): {X_test.shape}")

# Modul Identifikasi Konflik Entitas, Angka, Negara, Pure Lead, Institusi & Kota untuk Precision Gating
def get_conflict_flags(titles, contents):
    conflicts = np.zeros(len(titles), dtype=bool)
    for i in range(len(titles)):
        t_low = titles[i].lower()
        c_low = contents[i].lower()
        lead_low = ' '.join(c_low.split()[:45])
        
        # 1. Lokasi Global
        t_loc = get_canonical_locations(t_low)
        c_loc = get_canonical_locations(c_low)
        if t_loc and any(loc not in c_loc and not any(p in c_low for p in loc.split()) for loc in t_loc):
            conflicts[i] = True
            continue
            
        # 2. Tokoh Global
        t_fig = get_canonical_figures(t_low)
        c_fig = get_canonical_figures(c_low)
        if t_fig and any(f not in c_fig and not any(p in c_low for p in f.split()) for f in t_fig):
            conflicts[i] = True
            continue
            
        # 3. Angka / Numerik
        t_nums = extract_numbers_comprehensive(titles[i])
        c_nums = extract_numbers_comprehensive(contents[i])
        if t_nums and any(num not in c_nums for num in t_nums):
            conflicts[i] = True
            continue
            
        # 4. Dynamic Open-NER Conflicts (Role & Facility)
        m_r = ROLE_NAME_REGEX.findall(titles[i])
        if m_r and not any(r.lower() in c_low for r in m_r):
            conflicts[i] = True
            continue
            
        m_f = FACILITY_NAME_REGEX.findall(titles[i])
        if m_f and not any(f.lower() in c_low for f in m_f):
            conflicts[i] = True
            continue
            
        # 5. Country Conflicts (Inkongruensi Negara Asing)
        t_raw_c = COUNTRIES_PAT.findall(t_low)
        if t_raw_c:
            t_c = set(COUNTRIES_MAP[c] for c in t_raw_c)
            c_raw_c = COUNTRIES_PAT.findall(c_low)
            c_c = set(COUNTRIES_MAP[c] for c in c_raw_c)
            lead_raw_c = COUNTRIES_PAT.findall(lead_low)
            lead_c = set(COUNTRIES_MAP[c] for c in lead_raw_c)
            if (not t_c.intersection(c_c)) or (lead_c and not t_c.intersection(lead_c)):
                conflicts[i] = True
                continue
                
        # 6. Pure Lead Location & Figure Conflicts
        lead_loc_fast = get_canonical_locations(' '.join(c_low.split()[:40]))
        if t_loc and lead_loc_fast and not t_loc.intersection(lead_loc_fast):
            conflicts[i] = True
            continue
            
        lead_fig_fast = get_canonical_figures(' '.join(c_low.split()[:40]))
        if t_fig and lead_fig_fast and not t_fig.intersection(lead_fig_fast):
            conflicts[i] = True
            continue
            
        # 7. Security Rank Absence
        m_rk = RANK_REGEX.findall(titles[i])
        if m_rk and not any(r.lower() in c_low for r in m_rk):
            conflicts[i] = True
            continue
    return conflicts

train_conflicts = get_conflict_flags(train_titles, train_contents) | (sem_tr[0][:, 0] == 1.0) | (sem_tr[1][:, 0] == 1.0)
test_conflicts = get_conflict_flags(test_titles, test_contents) | (sem_te[0][:, 0] == 1.0) | (sem_te[1][:, 0] == 1.0)
print(f"Flag konflik teridentifikasi: Train={train_conflicts.sum()} ({train_conflicts.mean()*100:.1f}%) | Test={test_conflicts.sum()} ({test_conflicts.mean()*100:.1f}%)")


Mengekstrak 48 fitur teroptimasi (Fine-Grained Alignment & Sentences) untuk data train dan test...
X_train_raw shape: (14397, 48) -> X_train (49 Fitur Teroptimasi): (14397, 49)
X_test_raw shape : (3603, 48) -> X_test  (49 Fitur Teroptimasi): (3603, 49)
Flag konflik teridentifikasi: Train=5152 (35.8%) | Test=1260 (35.0%)


## 5. Stratified 5-Fold Cross Validation & Tri-Tree Soft Voting Ensemble (49 Fitur + Precision Gating $T_{cl}=0.760, T_{cf}=0.810$)

Pada tahap ini, kita melatih **Tri-Tree Soft Voting Ensemble** yang terdiri dari 3 model pohon berkemampuan tinggi dengan 49 fitur teroptimasi:
1. **XGBoost Classifier** (350 trees, max_depth=5, lr=0.035, subsample=0.8, colsample=0.8)
2. **HistGradientBoosting Classifier** (300 iterasi, max_depth=6, lr=0.035)
3. **ExtraTrees Classifier** (250 trees, max_depth=14)

Penggabungan dilakukan secara *Direct Probability Blending* (bobot seimbang $1/3$), dilanjutkan dengan kalibrasi *Multi-Objective Precision Gating* ($T_{clean} = 0.760, T_{conf} = 0.810$) untuk memaksimalkan Macro F1 hingga tembus **0.7346** dan menangkap **710 kasus hoaks**.

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

xgb_oof = np.zeros(len(train))
hgb_oof = np.zeros(len(train))
et_oof = np.zeros(len(train))

xgb_test = np.zeros(len(test))
hgb_test = np.zeros(len(test))
et_test = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Training Fold {fold+1}/5 (XGBoost, HistGradientBoosting, ExtraTrees) ---")
    
    # 1. XGBoost
    clf_xgb = xgb.XGBClassifier(
        n_estimators=350, max_depth=5, learning_rate=0.035,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED + fold,
        n_jobs=-1, eval_metric='logloss'
    )
    clf_xgb.fit(X_train[trn_idx], y_train[trn_idx])
    xgb_oof[val_idx] = clf_xgb.predict_proba(X_train[val_idx])[:, 1]
    xgb_test += clf_xgb.predict_proba(X_test)[:, 1] / 5.0
    
    # 2. HistGradientBoosting
    clf_hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.035, max_depth=6, random_state=SEED + fold
    )
    clf_hgb.fit(X_train[trn_idx], y_train[trn_idx])
    hgb_oof[val_idx] = clf_hgb.predict_proba(X_train[val_idx])[:, 1]
    hgb_test += clf_hgb.predict_proba(X_test)[:, 1] / 5.0
    
    # 3. ExtraTrees
    clf_et = ExtraTreesClassifier(
        n_estimators=250, max_depth=14, random_state=SEED + fold, n_jobs=-1
    )
    clf_et.fit(X_train[trn_idx], y_train[trn_idx])
    et_oof[val_idx] = clf_et.predict_proba(X_train[val_idx])[:, 1]
    et_test += clf_et.predict_proba(X_test)[:, 1] / 5.0

# Tri-Tree Soft Voting Ensemble Blend
blend_oof = (xgb_oof + hgb_oof + et_oof) / 3.0
blend_test = (xgb_test + hgb_test + et_test) / 3.0

# Evaluasi 1: Pencarian Optimal Threshold Tunggal
best_th, best_f1 = 0.760, 0.0
for th in np.arange(0.50, 0.90, 0.005):
    preds = (blend_oof >= th).astype(int)
    sc = f1_score(y_train, preds, average='macro')
    if sc > best_f1:
        best_f1, best_th = sc, th

print("\n" + "="*50)
print(f"HASIL KALIBRASI OPTIMAL THRESHOLD TUNGGAL: {best_th:.3f} | MACRO F1: {best_f1:.4f}")
print("="*50)
oof_preds = (blend_oof >= best_th).astype(int)
print(classification_report(y_train, oof_preds, digits=4))
print(f"Akurasi: {(oof_preds == y_train).mean()*100:.2f}%")
print("Confusion Matrix (Optimal Threshold Tunggal):")
print(confusion_matrix(y_train, oof_preds))

# Evaluasi 2: Multi-Objective Precision Gating (Dual-Threshold Pareto Tuning untuk F1 Maksimal)
# Memisahkan artikel fakta bersih vs artikel dengan indikasi konflik entitas/angka/negara/lead/institusi/kota
best_th_clean, best_th_conf = 0.760, 0.810
best_gating_f1 = 0.0
for th_cl in np.arange(0.72, 0.79, 0.005):
    for th_cf in np.arange(0.78, 0.84, 0.005):
        g_preds = np.where(train_conflicts, (blend_oof >= th_cf).astype(int), (blend_oof >= th_cl).astype(int))
        sc = f1_score(y_train, g_preds, average='macro')
        if sc > best_gating_f1:
            best_gating_f1 = sc
            best_th_clean, best_th_conf = th_cl, th_cf

print("\n" + "="*50)
print(f"HASIL KALIBRASI MULTI-OBJECTIVE PRECISION GATING: Th_clean={best_th_clean:.3f}, Th_conf={best_th_conf:.3f} | MACRO F1: {best_gating_f1:.4f}")
print("="*50)
oof_preds_gating = np.where(train_conflicts, (blend_oof >= best_th_conf).astype(int), (blend_oof >= best_th_clean).astype(int))
print(classification_report(y_train, oof_preds_gating, digits=4))
print(f"Akurasi Keseluruhan: {(oof_preds_gating == y_train).mean()*100:.2f}%")
print("Confusion Matrix (Multi-Objective Precision Gating):")
print(confusion_matrix(y_train, oof_preds_gating))

# Evaluasi 3: Kalibrasi Quantile Prior (~10% Kelas 0)
prior_pct = 9.98
q_th = np.percentile(blend_oof, prior_pct)
oof_preds_q = (blend_oof >= q_th).astype(int)
q_f1 = f1_score(y_train, oof_preds_q, average='macro')
print("\n" + "="*50)
print(f"HASIL KALIBRASI QUANTILE PRIOR (Top {prior_pct}%): Cutoff Prob={q_th:.4f} | MACRO F1: {q_f1:.4f}")
print("="*50)
print(classification_report(y_train, oof_preds_q, digits=4))


--- Training Fold 1/5 (XGBoost, HistGradientBoosting, ExtraTrees) ---
--- Training Fold 2/5 (XGBoost, HistGradientBoosting, ExtraTrees) ---
--- Training Fold 3/5 (XGBoost, HistGradientBoosting, ExtraTrees) ---
--- Training Fold 4/5 (XGBoost, HistGradientBoosting, ExtraTrees) ---
--- Training Fold 5/5 (XGBoost, HistGradientBoosting, ExtraTrees) ---

HASIL KALIBRASI OPTIMAL THRESHOLD TUNGGAL: 0.775 | MACRO F1: 0.7321
              precision    recall  f1-score   support

           0     0.5574    0.4767    0.5139      1437
           1     0.9429    0.9580    0.9504     12960

    accuracy                         0.9100     14397
   macro avg     0.7501    0.7174    0.7321     14397
weighted avg     0.9044    0.9100    0.9068     14397

Akurasi: 91.00%
Confusion Matrix (Optimal Threshold Tunggal):
[[  685   752]
 [  544 12416]]

HASIL KALIBRASI MULTI-OBJECTIVE PRECISION GATING: Th_clean=0.755, Th_conf=0.805 | MACRO F1: 0.7343
              precision    recall  f1-score   support

      

## 6. Prediksi Data Uji (`test.csv`) & Pembuatan Submission (Row-Independent Multi-Objective Gating)

Menerapkan kalibrasi Precision Gating ($T_{clean} = 0.760, T_{conf} = 0.810$) secara **100% murni dan independen per-baris (*row-independent*)** ke probabilitas ensemble data uji untuk menghasilkan file submission akhir yang mematuhi format resmi lomba.

In [7]:
# Menghasilkan Prediksi Data Uji dengan Precision Gating (Row-Independent)
test_preds = np.where(test_conflicts, (blend_test >= best_th_conf).astype(int), (blend_test >= best_th_clean).astype(int))

submission = pd.DataFrame({'id': test['id'], 'label': test_preds})
print("--- Distribusi Prediksi Submission (Precision Gating) ---")
print(submission['label'].value_counts())
print(f"Persentase Kelas 1: {(submission['label']==1).mean()*100:.2f}%")
print(f"Persentase Kelas 0: {(submission['label']==0).mean()*100:.2f}%")

submission.to_csv('submission_baseline.csv', index=False)
print("\nSubmission file saved successfully: submission_baseline.csv!")
print(f"Total rows: {len(submission)} | Null values: {submission.isnull().sum().sum()}")


--- Distribusi Prediksi Submission (Precision Gating) ---
label
1    3289
0     314
Name: count, dtype: int64
Persentase Kelas 1: 91.29%
Persentase Kelas 0: 8.71%

Submission file saved successfully: submission_baseline.csv!
Total rows: 3603 | Null values: 0
